In [1]:
import pandas as pd

In [2]:
CD = "..//data//16_diecasting.csv"  # 샷·실린더압력·주조압력·사이클타임·비스킷두께·형체력·상태

In [3]:
df = pd.read_csv(CD)

In [4]:
# ── [개념] 분포의 폭 — 최소·최대·범위 ───────────────────────────────────────
# 형식: s.min() / s.max() / 범위 = max - min
# 왜: 범위가 비정상적으로 크면 이상 신호. 사이클타임 범위 6149초는 정상 주조로 불가능(설비 잼).

In [5]:
pd.set_option("display.precision",1)

In [6]:
df["사이클타임"].min()

np.float64(20.6)

In [8]:
df["사이클타임"].max()

np.float64(6170.0)

In [9]:
df["사이클타임"].max() - df["사이클타임"].min()

np.float64(6149.4)

In [11]:
pd.set_option("display.precision",2)

In [12]:
# ── [개념] 중심 — 평균 vs 중앙값 ────────────────────────────────────────────
# 왜: 평균은 이상치에 끌려가고 중앙값은 강하다. 둘의 큰 격차 자체가 이상치 신호.
print(df["사이클타임"].mean(), df["사이클타임"].median())  # 64.75 22.6

64.75 22.6


In [13]:
df.columns

Index(['샷', '실린더압력', '주조압력', '사이클타임', '비스킷두께', '형체력', '상태'], dtype='str')

In [14]:
# ── [개념] 사분위수 — quantile ──────────────────────────────────────────────
# 형식: s.quantile(0.25)=Q1, 0.5=Q2(중앙값), 0.75=Q3.  인자는 0~1 비율.
# 실수: quantile(25)처럼 25를 넣으면 오류 — 0.25로.

In [15]:
df["사이클타임"].quantile(0.25)

np.float64(20.8)

In [17]:
print(df["사이클타임"].quantile([0.25, 0.5, 0.75])) # [20.8, 22.6, 35.925]

0.25    20.80
0.50    22.60
0.75    35.92
Name: 사이클타임, dtype: float64


In [18]:
df["사이클타임"].median()

np.float64(22.6)

In [19]:
35.92 - 20.80

15.120000000000001

In [20]:
df["사이클타임"].describe()

count     188.00
mean       64.75
std       450.13
min        20.60
25%        20.80
50%        22.60
75%        35.92
max      6170.00
Name: 사이클타임, dtype: float64

In [21]:
print(df["사이클타임"].mean() -  df["사이클타임"].median()) 

42.15


In [23]:
# ── [개념] 분포 요약 — describe / 여러 컬럼 비교 ────────────────────────────
# describe: count·mean·std·min·25%·50%·75%·max 를 한 번에. mean과 50%(중앙값)의 격차로 이상치 감지.
df[["실린더압력", "비스킷두께", "형체력"]].quantile([0.25, 0.5, 0.75])

,실린더압력,비스킷두께,형체력
0.25,215.75,11.0,255.0
0.50,218.00,13.0,258.0
0.75,265.00,17.0,363.0


In [24]:
# ── [실습 1] 구조·분포 살펴보기 (주조 전체) ───────────────────────────────── 10시 22분

In [25]:
df = pd.read_csv(CD)
print(df.head())
print(df.shape)  # (202, 7)
print(df.columns.tolist())
df.info()

   샷  실린더압력    주조압력  사이클타임  비스킷두께    형체력  상태
0  1  214.0  1037.0   20.7   10.0  258.0   0
1  2  217.0  1052.0   20.7   11.0  257.0   0
2  3  214.0  1037.0   20.8   11.0  254.0   0
3  4  217.0  1052.0   20.6   11.0  253.0   0
4  5  217.0  1052.0   20.6   11.0  254.0   0
(202, 7)
['샷', '실린더압력', '주조압력', '사이클타임', '비스킷두께', '형체력', '상태']
<class 'pandas.DataFrame'>
RangeIndex: 202 entries, 0 to 201
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   샷       202 non-null    int64  
 1   실린더압력   188 non-null    float64
 2   주조압력    188 non-null    float64
 3   사이클타임   188 non-null    float64
 4   비스킷두께   188 non-null    float64
 5   형체력     188 non-null    float64
 6   상태      202 non-null    int64  
dtypes: float64(5), int64(2)
memory usage: 11.2 KB


In [27]:
# ── [실습 2] 한 컬럼의 최소·최대·범위 (실린더압력) ────────────────────────── 10시 27분

In [28]:
print(df["실린더압력"].min())  # 108.0
print(df["실린더압력"].max())  # 265.0
print(round(df["실린더압력"].max() - df["실린더압력"].min(), 1))  # 157.0

108.0
265.0
157.0


In [29]:
# ── [실습 3] 정렬해서 이상치 후보 찾기 (사이클타임) ───────────────────────── 10시 32분

In [31]:
정렬 = df.sort_values("사이클타임", ascending=False)
정렬[["샷", "사이클타임", "상태"]].head()
# 맨 위 6170초 = 설비 잼(상태 1). 100초 넘는 샷도 이상 후보. 정상 사이클은 20~35초 부근.

,샷,사이클타임,상태
199,200,6170.0,1
180,181,652.3,1
146,147,125.9,0
181,182,93.1,1
190,191,91.2,1


In [33]:
# 10분간 휴식후 11시 15분까지 실습 4 진행

In [34]:
print(round(df["사이클타임"].mean(), 2))  # 64.75 (이상치에 끌림)
print(df["사이클타임"].median())  # 22.6
정상 = df[df["상태"] == 0]
print(round(정상["사이클타임"].mean(), 2))  # 27.67 (중앙값에 근접)

64.75
22.6
27.67


In [35]:
# ── [실습 5] quantile로 Q1·Q2·Q3 (실린더압력) ───────────────11시 20분까지

In [36]:
# ── [실습 5] quantile로 Q1·Q2·Q3 (실린더압력) ───────────────────────────────
print(df["실린더압력"].quantile(0.25))  # 215.75 (Q1)
print(df["실린더압력"].quantile(0.5))  # 218.0  (Q2 = 중앙값)
print(df["실린더압력"].quantile(0.75))  # 265.0  (Q3)

215.75
218.0
265.0


In [37]:
# ── [실습 6] describe로 평균-중앙값 격차 큰 컬럼 찾기 ────────────────────────

In [38]:
# 전치

In [39]:
import numpy as np

In [40]:
np.array([[5,0,77,3],[9,1,41,12],[5,77,3,23]])

array([[ 5,  0, 77,  3],
       [ 9,  1, 41, 12],
       [ 5, 77,  3, 23]])

In [41]:
np.array([[5,0,77,3],[9,1,41,12],[5,77,3,23]]).T

array([[ 5,  9,  5],
       [ 0,  1, 77],
       [77, 41,  3],
       [ 3, 12, 23]])

In [44]:
np.array([[5,0,77,3],[9,1,41,12],[5,77,3,23]]).transpose()

array([[ 5,  9,  5],
       [ 0,  1, 77],
       [77, 41,  3],
       [ 3, 12, 23]])

In [42]:
pd.DataFrame(np.array([[5,0,77,3],[9,1,41,12],[5,77,3,23]]))

,0,1,2,3
0,5,0,77,3
1,9,1,41,12
2,5,77,3,23


In [43]:
pd.DataFrame(np.array([[5,0,77,3],[9,1,41,12],[5,77,3,23]])).T

,0,1,2
0,5,9,5
1,0,1,77
2,77,41,3
3,3,12,23


In [45]:
pd.DataFrame(np.array([[5,0,77,3],[9,1,41,12],[5,77,3,23]])).T.transpose()

,0,1,2,3
0,5,0,77,3
1,9,1,41,12
2,5,77,3,23


In [49]:
# 아래 실습은 전치 활용해보기

In [48]:
# # ── [실습 6] describe로 평균-중앙값 격차 큰 컬럼 찾기 ────11시 43분까지────────────────────

In [50]:
df[["실린더압력", "주조압력", "사이클타임", "비스킷두께", "형체력"]].describe()

,실린더압력,주조압력,사이클타임,비스킷두께,형체력
count,188.00,188.00,188.00,188.00,188.00
mean,234.31,848.70,64.75,14.17,303.11
std,32.12,244.73,450.13,3.66,56.16
min,108.00,255.00,20.60,2.00,222.00
25%,215.75,595.00,20.80,11.00,255.00
50%,218.00,1037.00,22.60,13.00,258.00
75%,265.00,1052.00,35.92,17.00,363.00
max,265.00,1159.00,6170.00,27.00,384.00


In [51]:
df[["실린더압력", "주조압력", "사이클타임", "비스킷두께", "형체력"]].describe().T

,count,mean,std,min,25%,50%,75%,max
실린더압력,188.0,234.31,32.12,108.0,215.75,218.0,265.00,265.0
주조압력,188.0,848.70,244.73,255.0,595.00,1037.0,1052.00,1159.0
사이클타임,188.0,64.75,450.13,20.6,20.80,22.6,35.92,6170.0
비스킷두께,188.0,14.17,3.66,2.0,11.00,13.0,17.00,27.0
형체력,188.0,303.11,56.16,222.0,255.00,258.0,363.00,384.0


In [52]:
요약 = df[["실린더압력", "주조압력", "사이클타임", "비스킷두께", "형체력"]].describe().T

In [53]:
요약["격차"] = (요약["mean"] - 요약["50%"]).abs()

In [54]:
요약

,count,mean,std,min,25%,50%,75%,max,격차
실린더압력,188.0,234.31,32.12,108.0,215.75,218.0,265.00,265.0,16.31
주조압력,188.0,848.70,244.73,255.0,595.00,1037.0,1052.00,1159.0,188.30
사이클타임,188.0,64.75,450.13,20.6,20.80,22.6,35.92,6170.0,42.15
비스킷두께,188.0,14.17,3.66,2.0,11.00,13.0,17.00,27.0,1.17
형체력,188.0,303.11,56.16,222.0,255.00,258.0,363.00,384.0,45.11


In [56]:
요약.sort_values("격차", ascending=False)[["mean", "50%", "max"]].head(3)

,mean,50%,max
주조압력,848.70,1037.0,1159.0
형체력,303.11,258.0,384.0
사이클타임,64.75,22.6,6170.0


In [57]:
# 주조압력·형체력·사이클타임: 평균-중앙값 격차 큰 이상치 의심 컬럼